# Setup

In [ ]:
from pprint import pprint
import os, math, time
import pandas as pd
import torch
from transformers.utils import logging
from transformers import set_seed

# Device.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = device.type == 'cuda' and torch.cuda.is_bf16_supported()

# Suppress warnings.
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

# Seed.
seed = 42
set_seed(seed)

c:\Users\yana\Desktop\ai-summary\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0916 13:56:54.423000 17032 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


# Index

```markdown
1. Retrieval
  - Dense baseline                   
  - Sparse retrieval / BM25          
  - Hybrid                           
  - Cross-encoder reranking          
  - Compare retrieval metrics        
2. Context construction
3. LLM generation
4. Evaluation
5. Advanced vector search
```

# Terminology

## Steps

1. Parse and cleaning
2. Chunking

## Chunking

- Fixed length
- Structure-aware chunking
- Semantic chunking

## Prompt Engineering

- Query
  - Query rewriting: one better query.
  - Query expansion: enrich one query with additional terms.
  - Multi-query retrieval: create multiple alternative queries and combine.
  - Query decomposition: split complex query into simpler subqueries.
    - Multi-hop / iterative RAG
- HyDE: Hypothetical Document Embeddings
  - Instead of directly embedding the user query,
  - ask an LLM to generate a hypothetical answer/document,
  - and use this embedding for search.

## Retrieval

- Dense retrieval
  - Convert document/query into embeddings.
  - Retrieve the documents using similarity.
- Sparse retrieval (BM25)
  - Best Matching 25.
  - Search in the document, for the keywords in the query.
- Hybrid retrieval
  - Reciprocal Rank Fusion, $RRF(d) = \displaystyle \sum_r {\frac{1}{k + rank_r(d)}}$
- Metadata filtering
  - Filter by the given structured metadata.
- Top-$K$
  - Retrieve top-$K$ similar documents.
- Reranking
  - Bi-encoder
    - Embed query/document separately.
    - Search the document based on similarity and retrieve.
    - Recall: did we find related candidates in document?
  - Cross-encoder
    - Calculate relevance score of the query and retrieved candidates.
    - Query and candidates interact directly through attention.
    - Precision: are the candidates actually useful?

## Context Construction

- Ordering by score
- Deduplication
- Truncation according to the context budget
- Rearrange important evidences
  - Lost-in-the-middle problem: LLMs often treat the information near the beginning/end of a long context.
- Context compression
- Grounded generation
  - The answer should be supported by the retrieval, rather a pure LLM's internal knowledge.
- Abstention
  - If there is not enough evidence, explicitly says "I don't know" rather hallucination.
- Parent-child retrieval
  - Retrieve a child chunk -> add the whole parent chunk.

## RAG Evaluation

- Retriever
  - Recall@k
    - How much are relevant evidences in the top-k?
    - Total 5 relevences.
    - Top-3 = [A, e1, e2] -> 2/5
  - Hit rate@k
    - Is there at least one relevant evidence?
    - Top-3 = [A, e1, B] -> True = 1
  - Precision@k
    - What fraction of the top-k are relevant?
    - top-3 = [A, relevant, B] -> 1/3
  - MRR: Mean Reciprocal Rank
    - Rank of the first relevant evidence.
    - top-5 = [A, B, C, relevant, D] -> 4/5
  - nDCG: normalized Discounted Cumulative Gain
    - Evaluate the entire retrieved list -> more relevant should get higher rank.
  - Context relevance
    - Was the retrieved evidence actually useful?
- Generator
  - Answer quality
  - Faithfulness / groundness: is the answer supported by retrieved evidence?
  - Context relevance

## Advanced RAG

- Self-RAG
  - Self-Reflective RAG
  - Instead of always retrieving, the model learns to decide.
- CRAG
  - Corrective RAG
  - What should the LLM do when retrieved documents are poor?
- Graph RAG
  - Graph-based RAG
  - Stores an entity and relationship too.
  - Retrieve connected entities / relationships / supporting texts too.
- Agentic RAG
  - An agent actively controls retrieval.

## RAG Framework

- Qdrant: for general RAG system
  - `pip install qdrant-client`
- faiss: low-level api for vector search

# Dataset

- Dataset: SciFact
  - 5,183 corpus documents
  - 1,109 queries
  - 339 relevances (300 unique test queries)

In [2]:
from datasets import load_dataset

# Corpus.
corpus = load_dataset(
    'BeIR/scifact',
    'corpus',
    split='corpus',
)

# Queries.
queries = load_dataset(
    'BeIR/scifact',
    'queries',
    split='queries',
)

# Relevances. score = 1 -> relevant.
qrels = load_dataset(
    'BeIR/scifact-qrels',
    split='test',
)

# Example.
rel = qrels[0]
query_id = str(rel['query-id'])
corpus_id = str(rel['corpus-id'])

query = next(
    row for row in queries
    if str(row['_id']) == query_id
)
document = next(
    row for row in corpus
    if str(row['_id']) == corpus_id
)

print('Query:')
print(query['text'])

print('\nRelevant document:')
pprint(document['title'])
pprint(document['text'])

Query:
0-dimensional biomaterials show inductive properties.

Relevant document:
('New opportunities: the use of nanotechnologies to manipulate and track stem '
 'cells.')
('Nanotechnologies are emerging platforms that could be useful in measuring, '
 'understanding, and manipulating stem cells. Examples include magnetic '
 'nanoparticles and quantum dots for stem cell labeling and in vivo tracking; '
 'nanoparticles, carbon nanotubes, and polyplexes for the intracellular '
 'delivery of genes/oligonucleotides and protein/peptides; and engineered '
 'nanometer-scale scaffolds for stem cell differentiation and transplantation. '
 'This review examines the use of nanotechnologies for stem cell tracking, '
 'differentiation, and transplantation. We further discuss their utility and '
 'the potential concerns regarding their cytotoxicity.')


# 1. Retrieval

## Dense

### Embedding

- Sentence transformer
  - An embedding model designed to turn an entire sentence/document into one fixed-size embedding vector.
- `BAAI/bge-small-en-v1.5`
  - A small RAG embedding model for learning.
> Note) If an embedding is L2-normalized, i.e. $e' = \frac{e}{||e||_2}$, then $||e'||_2 = 1$ and $cos(e_q, e_d) = q^T d$

In [3]:
from sentence_transformers import SentenceTransformer

# Embedding model.
embedding_model = SentenceTransformer(
    'BAAI/bge-small-en-v1.5',
).to(device)

# Build one text string per document.
doc_texts = []

for row in corpus:
    title = row['title']
    text = row['text']

    doc_text = title + '\n' + text
    doc_texts.append(doc_text)

# Encode.
doc_embeddings = embedding_model.encode(
    doc_texts,
    batch_size=128,
    normalize_embeddings=True,      # L2-normalization.
    convert_to_numpy=True,
    show_progress_bar=True,
)

print(doc_embeddings.shape)     # (N, d) = (5_183, 384)

Batches: 100%|██████████| 41/41 [00:05<00:00,  7.56it/s]

(5183, 384)


### Client

```python
client = QdrantClient(
    path=...,           # persist in disk.
    path=':memory:',    # in-memory -> deleted after the process ends.
    url=...,            # qdrant server.
)
```

In [4]:
from qdrant_client import QdrantClient

# Client.
collection_name = 'scifact_dense'
client = QdrantClient(
    path='../tmp/outputs/qdrant_scifact',   # persist in disk.
)

### Collection

In [5]:
from qdrant_client import models

# Delete the collection if already exists.
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

# Create a collection.
vector_size = doc_embeddings.shape[1]       # (N, d)

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=vector_size,
        distance=models.Distance.COSINE,    # cosine similarity.
    )
)

True

### Upload

In [6]:
# One point.
row = corpus[0]
embedding = doc_embeddings[0]

point = models.PointStruct(
    id=int(row['_id']),
    vector=embedding.tolist(),
    payload={   # metadata.
        'title': row['title'],
        'text': row['text'],
    }
)

client.upsert(
    collection_name=collection_name,
    points=[point],
)

# Entire corpus.
points = []

for row, embedding in zip(corpus, doc_embeddings):
    point = models.PointStruct(
        id=int(row['_id']),
        vector=embedding.tolist(),
        payload={
            'title': row['title'],
            'text': row['text'],
        },
    )
    points.append(point)

client.upload_points(
    collection_name=collection_name,
    points=points,
    batch_size=128,
)

# Count.
count = client.count(
    collection_name=collection_name,
    exact=True,     # if False, use approximation.
)

print(f"Count: {count.count}")

# Close the client.
# client.close()

Count: 5183


### Query Embedding

In [7]:
query = queries[0]['text']

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

### Retrieval

In [8]:
results = client.query_points(
    collection_name=collection_name,
    query=query_embedding.tolist(),
    limit=3,                # top-3 documents.
    with_payload=True,      # returns the stored title/text too.
)
retrievals = results.points

for rank, result in enumerate(retrievals, start=1):
    print(f'Rank: {rank}')
    print(f'Doc ID: {result.id}')
    print(f'Score: {result.score:.4f}')
    print(f"Title: {result.payload['title']}")
    print(f"Text: {result.payload['text'][:50]} ...")
    print()

Rank: 1
Doc ID: 17388232
Score: 0.7618
Title: Mechanical regulation of cell function with geometrically modulated elastomeric substrates
Text: We report the establishment of a library of microm ...

Rank: 2
Doc ID: 4346436
Score: 0.7549
Title: Nonlinear Elasticity in Biological Gels
Text: Unlike most synthetic materials, biological materi ...

Rank: 3
Doc ID: 29638116
Score: 0.7102
Title: Complex Tissue and Disease Modeling using hiPSCs.
Text: Defined genetic models based on human pluripotent  ...



### Evaluation

In [22]:
import ir_measures
from ir_measures import R, RR, nDCG, Success

# 1. Test queries.
test_query_ids = sorted({
    str(row['query-id'])
    for row in qrels
})
query_lookup = {
    str(row['_id']): row['text']
    for row in queries
}
test_query_texts = [
    query_lookup[query_id]
    for query_id in test_query_ids
]
print('Test queries:', len(test_query_ids))

# 2. Embedding.
query_embeddings = embedding_model.encode(
    test_query_texts,
    batch_size=128,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

# 3. Retrieval.
requests = [
    models.QueryRequest(
        query=embedding.tolist(),
        limit=5,
        with_payload=False,
    )
    for embedding in query_embeddings
]
responses = client.query_batch_points(
    collection_name=collection_name,
    requests=requests,
)

# 4. Ground truth -> ir_measures.
eval_qrels = [
    ir_measures.Qrel(
        query_id=str(row['query-id']),
        doc_id=str(row['corpus-id']),
        relevance=int(row['score']),
    )
    for row in qrels
]

# 5. Response -> ir_measures.
run = []
for query_id, response in zip(test_query_ids, responses):
    for point in response.points:
        run.append(
            ir_measures.ScoredDoc(
                query_id=query_id,
                doc_id=str(point.id),
                score=float(point.score),
            )
        )

# 6. Evaluate.
dense_metrics = ir_measures.calc_aggregate(
    [
        Success@5,      # hit rate@5.
        R@5,            # recall@5.
        R@10,           # recall@10.
        RR@10,          # mrr@10.
        nDCG@10,        # nDCG.
    ],
    eval_qrels,
    run,
)

for metric, value in dense_metrics.items():
    print(f'{metric}: {value:.4f}')

Test queries: 300


Batches: 100%|██████████| 3/3 [00:00<00:00, 17.52it/s]


RR@10: 0.6743
R@10: 0.7653
nDCG@10: 0.6922
Success@5: 0.7833
R@5: 0.7653


## Sparse / Hybrid

- `fastemb`
  - A lightweight embedding library of `qdrant`.
  - Supports a sparse embedding like `bm25`.
- Sparse vector
  - A vector where almost all dimensions are zero.
  - Dense: <0, 0.2, 0, 0, 0.5>
  - Sparse representation: indices=[1, 4], values=[0.2, 0.5]
- Sparse representation
  - Represent words/terms as a sparse vector.
  - values = $\mathrm{BM25}(Q,D)=\sum_{q_i\in Q}\mathrm{IDF}(q_i)\cdot\frac{f(q_i,D)(k_1+1)}{f(q_i,D)+k_1\left(1-b+b\frac{|D|}{\mathrm{avgdl}}\right)}$
    - $Q$: query.
    - $D$: document.
    - $q_i$: one term in the query.
    - $f(q_i,D)$: frequency of query term $q_i$ in document $D$.
    - $k_1$: controls term-frequency saturation.
      - Larger $k_1$ -> higher score on repeated occurrence.
    - $b$: controls document-length normalization.
      - $b=0$ → ignore document length.
      - $b=1$ → fully apply length normalization.
    - $|D|$: length of the current document.
    - $\mathrm{avgdl}$: average document length in the corpus.
    - $\mathrm{IDF}(q_i)$: Inverse Document Frequency of term $q_i$.
  - $\mathrm{IDF}(q_i)=\ln\left(\frac{N-n(q_i)+0.5}{n(q_i)+0.5}+1\right)$
    - A rareness of the term in documents.
    - $N$: total number of documents in the corpus.
    - $n(q_i)$: number of documents containing term $q_i$.
    - Rare terms → high IDF.
    - Common terms → low IDF.
- Hybrid retrieval
  - Dense vectors: for meaning
  - Sparse vectors: for words/terms.

### Embedding

In [10]:
from fastembed import SparseTextEmbedding

# Model.
sparse_model = SparseTextEmbedding(
    model_name='Qdrant/bm25',
    language='english',
)

# Embedding.
sparse_embedding_iter = sparse_model.embed(
    doc_texts,
    batch_size=256,
)
first_embedding = next(sparse_embedding_iter)

print(f"Total documents: {len(doc_texts):,}")
print(f"Indices: {first_embedding.indices[:5]}")
print(f"Values: {first_embedding.values[:5]}")

Total documents: 5,183
Indices: [1906054390  659326392 1082468256 1232530885  242156862]
Values: [1.4074722  1.79557483 1.4074722  1.0347235  1.79557483]


### Point Structure

```markdown
Point
├─ id
├─ dense vector
├─ sparse vector
└─ payload
```

In [11]:
# Iterator.
def point_iter():
    for row, dense_embedding, sparse_embedding in zip(
        corpus,
        doc_embeddings,
        sparse_embedding_iter,
    ):
        yield models.PointStruct(
            id=int(row['_id']),
            vector={
                'dense': dense_embedding.tolist(),
                'sparse': models.SparseVector(
                    indices=sparse_embedding.indices.tolist(),
                    values=sparse_embedding.values.tolist(),
                ),
            },
            payload={
                'title': row['title'],
                'text': row['text'],
            },
        )

### Collection

In [12]:
hybrid_collection = 'scifact_hybrid'

# Delete if exists.
if client.collection_exists(hybrid_collection):
    client.delete_collection(hybrid_collection)

# Create.
client.create_collection(
    collection_name=hybrid_collection,
    vectors_config={
        'dense': models.VectorParams(
            size=doc_embeddings.shape[1],
            distance=models.Distance.COSINE,
        ),
    },
    sparse_vectors_config={
        'sparse': models.SparseVectorParams(
            # don't need a fixed size.
            modifier=models.Modifier.IDF,
        ),
    },
)

True

### Upload

In [13]:
# Recreate from beginning.
sparse_embedding_iter = sparse_model.embed(
    doc_texts,
    batch_size=256,
)

# Upload.
client.upload_points(
    collection_name=hybrid_collection,
    points=point_iter(),
    batch_size=128,
)

# Count.
count = client.count(
    collection_name=hybrid_collection,
    exact=True,
)
print(f"Count: {count.count:,}")

Count: 5,183


### Query Embedding

In [14]:
query = queries[0]['text']

sparse_query = next(
    sparse_model.query_embed(query)
)

pprint(sparse_query)

SparseEmbedding(values=array([1, 1, 1, 1, 1, 1], dtype=int32),
                indices=array([ 764297089, 1353987491, 1701701189,  834323496, 1678399785,
       1861059290], dtype=int32))


### Retrieval

#### Sparse

In [15]:
results = client.query_points(
    collection_name=hybrid_collection,
    query=models.SparseVector(
        indices=sparse_query.indices.tolist(),
        values=sparse_query.values.tolist(),
    ),
    using='sparse',     # vector name in config.
    limit=3,
    with_payload=True,
)
retrievals = results.points

for rank, result in enumerate(retrievals, start=1):
    print(f'Rank: {rank}')
    print(f'Doc ID: {result.id}')
    print(f'Score: {result.score:.4f}')
    print(f"Title: {result.payload['title']}")
    print(f"Text: {result.payload['text'][:50]} ...")
    print()

Rank: 1
Doc ID: 26071782
Score: 11.7429
Title: Latent membrane protein 1 of Epstein–Barr virus coordinately regulates proliferation with control of apoptosis
Text: Latent membrane protein 1 (LMP1), an oncoprotein e ...

Rank: 2
Doc ID: 10608397
Score: 10.6984
Title: High-performance neuroprosthetic control by an individual with tetraplegia.
Text: BACKGROUND Paralysis or amputation of an arm resul ...

Rank: 3
Doc ID: 21257564
Score: 10.4448
Title: How long will long-term potentiation last?
Text: The paramount feature of long-term potentiation (L ...



#### Hybrid

- Reciprocal Rank Fusion, $RRF(d) = \displaystyle \sum_r {\frac{1}{k + rank_r(d)}}$
  - Small k → top ranks matter more.
  - Large k → rankings are smoothed.

In [16]:
# Encode query.
query = queries[0]['text']

dense_query = embedding_model.encode(
    query,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

sparse_query = next(
    sparse_model.query_embed(query)
)

# Retrieve.
results = client.query_points(
    collection_name=hybrid_collection,
    prefetch=[
        models.Prefetch(    # find 10 cands using dense.
            query=dense_query.tolist(),
            using='dense',
            limit=10,
        ),
        models.Prefetch(    # find 10 cands using sparse.
            query=models.SparseVector(
                indices=sparse_query.indices.tolist(),
                values=sparse_query.values.tolist(),
            ),
            using='sparse',
            limit=10,
        ),
    ],
    query=models.FusionQuery(
        fusion=models.Fusion.RRF,
    ),
    limit=3,
    with_payload=True,
)
retrievals = results.points

for rank, result in enumerate(retrievals, start=1):
    print(f'Rank: {rank}')
    print(f'Doc ID: {result.id}')
    print(f'Score: {result.score:.4f}')
    print(f"Title: {result.payload['title']}")
    print()

Rank: 1
Doc ID: 17388232
Score: 0.5000
Title: Mechanical regulation of cell function with geometrically modulated elastomeric substrates

Rank: 2
Doc ID: 26071782
Score: 0.5000
Title: Latent membrane protein 1 of Epstein–Barr virus coordinately regulates proliferation with control of apoptosis

Rank: 3
Doc ID: 40212412
Score: 0.3667
Title: Periosteal bone formation--a neglected determinant of bone strength.



### Evaluation

#### Sparse

In [17]:
# Request.
sparse_requests = []
for sparse_embedding in sparse_model.query_embed(test_query_texts):
    request = models.QueryRequest(
        query=models.SparseVector(
            indices=sparse_embedding.indices.tolist(),
            values=sparse_embedding.values.tolist(),
        ),
        using='sparse',
        limit=5,
        with_payload=False,
    )
    sparse_requests.append(request)

# Response.
sparse_responses = client.query_batch_points(
    collection_name=hybrid_collection,
    requests=sparse_requests,
)

# Evaluate.
sparse_run = []
for query_id, response in zip(test_query_ids, sparse_responses):
    for point in response.points:
        sparse_run.append(
            ir_measures.ScoredDoc(
                query_id=query_id,
                doc_id=str(point.id),
                score=float(point.score),
            )
        )

sparse_metrics = ir_measures.calc_aggregate(
    [
        Success@5,
        R@5,
        R@10,
        RR@10,
        nDCG@10,
    ],
    eval_qrels,
    sparse_run,
)

for metric, value in sparse_metrics.items():
    print(f'{metric}: {value:.4f}')

RR@10: 0.6381
R@10: 0.7459
nDCG@10: 0.6610
Success@5: 0.7633
R@5: 0.7459


#### Hybrid

In [18]:
# Request.
hybrid_requests = []
for query_text, dense_embedding in zip(
    test_query_texts,
    query_embeddings,
):
    sparse_embedding = next(
        sparse_model.query_embed(query_text)
    )

    request = models.QueryRequest(
        prefetch=[
            models.Prefetch(
                query=dense_embedding.tolist(),
                using='dense',
                limit=5,
            ),
            models.Prefetch(
                query=models.SparseVector(
                    indices=sparse_embedding.indices.tolist(),
                    values=sparse_embedding.values.tolist(),
                ),
                using='sparse',
                limit=5,
            ),
        ],
        query=models.FusionQuery(
            fusion=models.Fusion.RRF,
        ),
        limit=5,
        with_payload=False,
    )

    hybrid_requests.append(request)

# Response.
hybrid_responses = client.query_batch_points(
    collection_name=hybrid_collection,
    requests=hybrid_requests,
)

# Evaluate.
hybrid_run = []
for query_id, response in zip(
    test_query_ids,
    hybrid_responses,
):
    for point in response.points:
        hybrid_run.append(
            ir_measures.ScoredDoc(
                query_id=query_id,
                doc_id=str(point.id),
                score=float(point.score),
            )
        )

hybrid_metrics = ir_measures.calc_aggregate(
    [
        Success@5,
        R@5,
        R@10,
        RR@10,
        nDCG@10,
    ],
    eval_qrels,
    hybrid_run,
)

for metric, value in hybrid_metrics.items():
    print(f'{metric}: {value:.4f}')

RR@10: 0.6954
R@10: 0.7845
nDCG@10: 0.7064
Success@5: 0.8000
R@5: 0.7845


## Reranker

- Bi-encoder
  - query -> embedding
  - document -> embedding
  - compare embedding vectors
- Cross encoder
  - A transformer model for (query, document) -> embedding -> relevance score.
  - Because a query/document attends each other, it is more accurate but slower.

### Sample

In [19]:
from sentence_transformers import CrossEncoder

# Create.
reranker = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L6-v2',
    device=device,
)

# Example.
pairs = [
    ['What causes cancer?', 'Cancer is caused by genetic mutations ...'],
    ['What causes cancer?', 'The weather is sunny today.'],
]
scores = reranker.predict(
    pairs,
    batch_size=128,
)
print(f"Scores: {scores}")

Scores: [  6.801113 -11.118631]


### Evaluation

In [21]:
# 1. Hybrid retrieval: get top-10 candidates per query.
candidate_k = 20
sparse_query_iter = sparse_model.query_embed(test_query_texts)
rerank_requests = []

for dense_embedding, sparse_embedding in zip(
    query_embeddings,
    sparse_query_iter,
):
    request = models.QueryRequest(
        prefetch=[
            models.Prefetch(
                query=dense_embedding.tolist(),
                using='dense',
                limit=candidate_k,
            ),
            models.Prefetch(
                query=models.SparseVector(
                    indices=sparse_embedding.indices.tolist(),
                    values=sparse_embedding.values.tolist(),
                ),
                using='sparse',
                limit=candidate_k,
            ),
        ],
        query=models.FusionQuery(
            fusion=models.Fusion.RRF,
        ),
        limit=candidate_k,
        with_payload=True,
    )

    rerank_requests.append(request)

candidate_responses = client.query_batch_points(
    collection_name=hybrid_collection,
    requests=rerank_requests,
)


# 2. Build [query, document] pairs for the cross-encoder.
pairs = []
pair_ids = []

for query_id, query_text, response in zip(
    test_query_ids,
    test_query_texts,
    candidate_responses,
):
    for point in response.points:
        document = (
            point.payload['title']
            + '\n'
            + point.payload['text']
        )
        pairs.append((query_text, document))
        pair_ids.append((query_id, str(point.id)))


# 3. Score every query-document pair with the cross-encoder.
rerank_scores = reranker.predict(
    pairs,
    batch_size=256,
    show_progress_bar=True,
)


# 4. Convert reranker outputs to ir_measures format.
reranked_run = []

for (query_id, doc_id), score in zip(
    pair_ids,
    rerank_scores,
):
    reranked_run.append(
        ir_measures.ScoredDoc(
            query_id=query_id,
            doc_id=doc_id,
            score=float(score),
        )
    )


# 5. Evaluate the reranked results.
reranked_metrics = ir_measures.calc_aggregate(
    [
        Success@5,
        R@5,
        R@10,
        RR@10,
        nDCG@10,
    ],
    eval_qrels,
    reranked_run,
)

for metric, value in reranked_metrics.items():
    print(f'{metric}: {value:.4f}')

Batches: 100%|██████████| 24/24 [00:03<00:00,  6.93it/s]


RR@10: 0.6699
R@10: 0.8384
nDCG@10: 0.7037
Success@5: 0.7867
R@5: 0.7649


## Comparison

In [23]:
results_df = pd.DataFrame({
    'Dense': dense_metrics,
    'Sparse': sparse_metrics,
    'Hybrid': hybrid_metrics,
    'Reranked': reranked_metrics,
}).T

display(results_df)

,RR@10,R@10,nDCG@10,Success@5,R@5
Dense,0.674333,0.765278,0.692210,0.783333,0.765278
Sparse,0.638056,0.745944,0.661047,0.763333,0.745944
Hybrid,0.695444,0.784500,0.706429,0.800000,0.784500
Reranked,0.669950,0.838444,0.703741,0.786667,0.764889


# 2. Context Construction

In [42]:
# n_contexts.
context_k = 3

# Query.
query_id = test_query_ids[0]

# Corpus lookup.
corpus_lookup = {
    str(row['_id']): row
    for row in corpus
}

# Retrieved docs.
top_docs = sorted(
    [
        (doc_id, score)
        for (qid, doc_id), score in zip(pair_ids, rerank_scores)
        if qid == query_id
    ],
    key=lambda x: x[1],
    reverse=True,
)[:context_k]

# Context.
context = '\n\n'.join(
    f"[Document {i} | ID={doc_id}]\n"
    f"- Title: {corpus_lookup[doc_id]['title']}\n"
    f"- Content: {corpus_lookup[doc_id]['text'][:100]} ..."
    for i, (doc_id, score) in enumerate(top_docs, start=1)
)

print(f"Context: \n{context}")

Context: 
[Document 1 | ID=43385013]
- Title: Epithelial and mesenchymal subpopulations within normal basal breast cell lines exhibit distinct stem cell/progenitor properties.
- Content: It has been proposed that epithelial-mesenchymal transition (EMT) in mammary epithelial cells and br ...

[Document 2 | ID=10608397]
- Title: High-performance neuroprosthetic control by an individual with tetraplegia.
- Content: BACKGROUND Paralysis or amputation of an arm results in the loss of the ability to orient the hand a ...

[Document 3 | ID=27049238]
- Title: Large deformation of red blood cell ghosts in a simple shear flow.
- Content: Red blood cells are known to change shape in response to local flow conditions. Deformability affect ...


# 3. LLM Generation

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Model.
generator_name = 'Qwen/Qwen2.5-0.5B-Instruct'
generator_tokenizer = AutoTokenizer.from_pretrained(generator_name)
generator_model = AutoModelForCausalLM.from_pretrained(
    generator_name,
    torch_dtype=torch.bfloat16,
).to(device)

# Prompt.
query_id = 42
query_text = query_lookup[str(query_id)]
messages = [
    {
        'role': 'system',
        'content': (
            'Determine whether the claim is Supported, Refuted, or Unknown '
            'based only on the provided context. '
            'Cite the supporting document IDs. '
            'Do not use outside knowledge.'
        ),
    },
    {
        'role': 'user',
        'content': (
            f'Claim: {query_text}\n\n'
            f'Context:\n{context}'
        ),
    },
]

prompt = generator_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = generator_tokenizer(
    prompt,
    return_tensors='pt',
).to(device)

# Inference.
with torch.inference_mode():
    output_ids = generator_model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
    )

generated_ids = output_ids[:, inputs['input_ids'].shape[1]:]

answer = generator_tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True,
)

print(f"{messages[1]['content']}")
print(f"\nAnswer: {answer}")

Claim: A high microerythrocyte count raises vulnerability to severe anemia in homozygous alpha (+)- thalassemia trait subjects.

Context:
[Document 1 | ID=43385013]
- Title: Epithelial and mesenchymal subpopulations within normal basal breast cell lines exhibit distinct stem cell/progenitor properties.
- Content: It has been proposed that epithelial-mesenchymal transition (EMT) in mammary epithelial cells and br ...

[Document 2 | ID=10608397]
- Title: High-performance neuroprosthetic control by an individual with tetraplegia.
- Content: BACKGROUND Paralysis or amputation of an arm results in the loss of the ability to orient the hand a ...

[Document 3 | ID=27049238]
- Title: Large deformation of red blood cell ghosts in a simple shear flow.
- Content: Red blood cells are known to change shape in response to local flow conditions. Deformability affect ...

Answer: Refuted


# 4. Evaluation

- LLM-as-a-judge
  - Use somewhat stronger model.

In [62]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Model.
judge_name = 'Qwen/Qwen2.5-1.5B-Instruct'
judge_tokenizer = AutoTokenizer.from_pretrained(judge_name)
judge_model = AutoModelForCausalLM.from_pretrained(
    judge_name,
    torch_dtype=torch.bfloat16,
).to(device)

# Prompt.
judge_messages = [
    {
        'role': 'system',
        'content': (
            'Evaluate whether the generated answer is supported by the provided context. '
            'Do not use outside knowledge. '
            'Return one of: Grounded, Partially Grounded, Ungrounded. '
            'Give a short reason.'
        ),
    },
    {
        'role': 'user',
        'content': (
            f'Claim:\n{query_text}\n\n'
            f'Context:\n{context}\n\n'
            f'Generated answer:\n{answer}'
        ),
    },
]

judge_prompt = judge_tokenizer.apply_chat_template(
    judge_messages,
    tokenize=False,
    add_generation_prompt=True,
)

judge_inputs = judge_tokenizer(
    judge_prompt,
    return_tensors='pt',
).to(device)

# Evaluate.
with torch.inference_mode():
    judge_output = judge_model.generate(
        **judge_inputs,
        max_new_tokens=64,
        do_sample=False,
    )

judge_generated = judge_output[:, judge_inputs['input_ids'].shape[1]:]

judgment = judge_tokenizer.decode(
    judge_generated[0],
    skip_special_tokens=True,
)

pprint(f"Result: \n{judge_messages}")
print("")
pprint(f"judgment: \n{judgment}")

('Result: \n'
 "[{'role': 'system', 'content': 'Evaluate whether the generated answer is "
 'supported by the provided context. Do not use outside knowledge. Return one '
 "of: Grounded, Partially Grounded, Ungrounded. Give a short reason.'}, "
 "{'role': 'user', 'content': 'Claim:\\nA high microerythrocyte count raises "
 'vulnerability to severe anemia in homozygous alpha (+)- thalassemia trait '
 'subjects.\\n\\nContext:\\n[Document 1 | ID=43385013]\\n- Title: Epithelial '
 'and mesenchymal subpopulations within normal basal breast cell lines exhibit '
 'distinct stem cell/progenitor properties.\\n- Content: It has been proposed '
 'that epithelial-mesenchymal transition (EMT) in mammary epithelial cells and '
 'br ...\\n\\n[Document 2 | ID=10608397]\\n- Title: High-performance '
 'neuroprosthetic control by an individual with tetraplegia.\\n- Content: '
 'BACKGROUND Paralysis or amputation of an arm results in the loss of the '
 'ability to orient the hand a ...\\n\\n[Document 3 | 

# 5. Vector Search

## Exact Search

- Calculate the score of all $(q, d)$ pairs.

In [69]:
exact_results = client.query_points(
    collection_name=hybrid_collection,
    query=query_embeddings[0].tolist(),
    using='dense',
    search_params=models.SearchParams(  
        exact=True,     # exact search.
    ),
    limit=10,
    with_payload=False,
)

## HNSW

- HNSW: Hierarchical Navigable Small Word graph

### Algorithm

```markdown
1. Start from the entry point `E` in `L_max`.
2. Compare the query `q` with `E` and its neighbors.
3. If a neighbor has a better similarity score,
  - move to the best such neighbor.
4. Repeat until no neighbor improves the current score.
5. Descend one layer, starting from that same current node.
  - if best has fewer than ef nodes,
  - or neighbor is better than the worst node in best,
  - add neighbor to candidates / best
6. Repeat steps 2–5 until reaching layer 0.
7. At layer 0, perform a broader graph search.
8. Return `top-k` from best.
```

### Broader graph search

```markdown
1. Put e₀ into candidates and best.
2. Take the highest-similarity node from candidates.
3. Look at all of that node's graph neighbors.
4. For each unvisited neighbor:
    compute similarity(q, neighbor)
5. If that neighbor is good enough to belong among the current best `ef` nodes:
    add it to candidates
    add it to best
6. If best contains more than `ef` nodes:
    remove its worst-scoring node.
7. Go back to step 2 and explore the next most promising candidate.
8. Stop when the best unexplored candidate cannot improve
   the current best set.
9. Return best.
```

### Parameters

```markdown
- `m`: edges per node
  - small `m`
    - fewer graph connections per node
    - lower memory usage
    - faster index construction
    - easier to get trapped in poor regions
    - potentially lower recall
  - large `m`
    - more graph connections per node
    - higher memory usage
    - slower index construction
    - more alternative paths through the graph
    - usually higher recall

- `ef_construct`: construction-time search breadth
  - small `ef_construct`
    - narrow search when choosing neighbors during index construction
    - faster index build
    - lower-quality graph connections
    - potentially lower recall
  - large `ef_construct`
    - broader search when choosing neighbors during index construction
    - slower index build
    - higher-quality graph connections
    - usually higher recall

- `hnsw_ef`: query-time search breadth
  - small `hnsw_ef`
    - narrow exploration
    - faster
    - easier to miss good regions
  - large `hnsw_ef`
    - broader exploration
    - slower
    - closer to exact search
```

### Example

> Note) a `qdrant` client in local only supports `exact` search.

In [70]:
# Suppose the client is on the server.

# Collection.
hnsw_collection = 'hnsw'

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=hnsw_collection,
    vectors_config={
        'dense': models.VectorParams(
            size=doc_embeddings.shape[1],
            distance=models.Distance.COSINE,
            hnsw_config=models.HnswConfigDiff(
                m=16,
                ef_construct=100,
            ),
        ),
    },
    sparse_vectors_config={
        'sparse': models.SparseVectorParams(
            modifier=models.Modifier.IDF,
        ),
    },
)

# Retrieval.
results = client.query_points(
    collection_name=hnsw_collection,
    query=query_embedding.tolist(),
    using='dense',
    search_params=models.SearchParams(
        hnsw_ef=128,
        exact=False,
    ),
    limit=10,
)

## FAISS: Advanced Search

- FlatIP: exact search
- HNSW: hierarchical graph
- IVF: clustering vectors
- PQ: product quantization
- IVF-PQ: IVF + PQ

### Setup

In [71]:
import faiss
import numpy as np

xb = np.asarray(doc_embeddings, dtype='float32')        # document vectors.
xq = np.asarray(query_embeddings, dtype='float32')      # query vectors.
d  = xb.shape[1]

### FlatIP

- Flat: search all stored vectors.
- IP: inner product
  - since the document vectors are L2-normalized, the cosine similarity equals to the inner product.

In [72]:
index_flat = faiss.IndexFlatIP(d)
index_flat.add(xb)

### HNSW

- See the above `HNSW` chapter.

In [73]:
index_hnsw = faiss.IndexHNSWFlat(
    d,
    16,  # M
    faiss.METRIC_INNER_PRODUCT,
)

index_hnsw.hnsw.efConstruction = 100
index_hnsw.hnsw.efSearch = 64
index_hnsw.add(xb)

### IVF

- Inverted File Index.
- It reduces the search space by partitioning vectors into clusters.
- Training
  - runs clustering (e.g. k-means) over document vectors, where `nlist` = n_clusters.
  - each cluster is called an 'inverted list'.
  - each document is assigned to the nearest.
- Query
  - compare `q` against `nlist` centroids.
  - `nprobe`: find in the closest `nprobe` clusters.

In [74]:
nlist = 64
ivf_quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVFFlat(
    ivf_quantizer,
    d,
    nlist,
    faiss.METRIC_INNER_PRODUCT,
)

index_ivf.train(xb)
index_ivf.add(xb)
index_ivf.nprobe = 8

### PQ

- Compress a document vector using Product Quantization (PQ).
  - original vector with $d = 384$ dimensions
  - split into $m = 16$ subchunks with $d_{sub} = 24$ each
  - all first subchunks across database vectors are clustered into $2^{nbits}$ clusters
  - with $nbits = 8$, each subspace has $2^8 = 256$ centroids and each centroid ID requires 1 byte
  - all other subchunks are clustered separately
- Codebooks
  - subspace 1 -> 256 representative centroids
  - ...
  - subspace 16 -> 256 representative centroids
- Full vector approximation
  - centroid(chunk 1) $\oplus$ centroid(chunk 2) $\oplus$ ... $\oplus$ centroid(chunk 16)
  - $\oplus$ means concatenation
- Vector size
  - Original: 384 x 4 bytes = 1,536 bytes/vector
  - PQ code: 16 x 1 byte = 16 bytes/vector
- Query
  - ADC: Asymmetric Distance Computation
    - database vectors are quantized
    - query vector remains uncompressed
  - split the query $q = q_1, q_2, ..., q_{16}$
  - calculate the score between each $q_i$ and all 256 centroids in that subspace
  - build $LUT$ with shape $(16, 256)$
  - for document $d$:
    - $\text{score}(q,d) \approx \displaystyle \sum_i LUT_i[d[i]]$
    - $d[i]$ is the centroid ID stored by document $d$ for subspace $i$

In [75]:
m = 16
nbits = 8
index_pq = faiss.IndexPQ(
    d,
    m,
    nbits,
    faiss.METRIC_INNER_PRODUCT,
)

index_pq.train(xb)
index_pq.add(xb)

### IVF-PQ

- Simply combines IVF + PQ
- But `faiss`'s `IndexIVFPQ` compresses the residual:

```markdown
x
→ IVF cluster centroid c
→ residual r = x - c
→ split r into m subchunks
→ PQ encode each subchunk
```

In [76]:
ivfpq_quantizer = faiss.IndexFlatIP(d)
index_ivfpq = faiss.IndexIVFPQ(
    ivfpq_quantizer,
    d,
    nlist,
    m,
    nbits,
    faiss.METRIC_INNER_PRODUCT,
)

index_ivfpq.train(xb)
index_ivfpq.add(xb)
index_ivfpq.nprobe = 8

## Comparison

```markdown
RAG Recall@10
→ how many gold relevant documents were retrieved?

ANN Recall@10
→ how many exact nearest neighbors were recovered
   compared with Flat search?
```

In [ ]:
k = 10
indexes = {
    'Flat': index_flat,
    'HNSW': index_hnsw,
    'IVF-Flat': index_ivf,
    'PQ': index_pq,
    'IVF-PQ': index_ivfpq,
}

# Ground truth: Flat.
_, exact_indices = index_flat.search(xq, k)

# Evaluate.
results = []
for name, index in indexes.items():

    start = time.perf_counter()
    _, retrieved_indices = index.search(xq, k)
    search_time = time.perf_counter() - start

    recall = sum(
        len(set(exact_indices[i]) & set(retrieved_indices[i])) / k
        for i in range(len(xq))
    ) / len(xq)

    memory_mb = len(faiss.serialize_index(index)) / (1024 ** 2)

    results.append({
        'Index': name,
        'Recall@10': recall,
        'Latency (ms/query)': search_time / len(xq) * 1000,
        'Index Size (MB)': memory_mb,
    })

results_df = pd.DataFrame(results)

display(results_df)

,Index,Recall@10,Latency (ms/query),Index Size (MB)
0,Flat,1.000000,0.040790,7.592328
1,HNSW,0.996333,0.007354,8.303755
2,IVF-Flat,0.896000,0.005528,7.726199
3,PQ,0.468000,0.016061,0.454168
4,IVF-PQ,0.473333,0.005619,0.588039
